### Contextual Compression Retriever (ContextualCompressionRetriever)

Standard vector retrieval often returns document chunks containing extra irrelevant noise surrounding the relevant information.

ContextualCompressionRetriever wraps a base retriever and a document compressor/filter to post-process retrieved context before returning it to the LLM.

### Key Components

* Base Retriever: Underlying retriever (e.g. Chroma vector store retriever).
* EmbeddingsFilter: Evaluates exact vector similarity between query and retrieved document chunks, filtering out low-relevance documents.
* LLMChainExtractor: Uses an LLM to extract only relevant sentences from retrieved documents.

In [1]:
from dotenv import load_dotenv, find_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter
from langchain_core.documents import Document

load_dotenv(find_dotenv())

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

docs = [
    Document(page_content="Quantum computing leverages superposition and entanglement to solve complex optimization problems."),
    Document(page_content="Classical computers process binary bits 0 or 1 using silicon transistors."),
    Document(page_content="French pastries include croissants, pain au chocolat, and macarons baked with butter.")
]

# Ephemeral in-memory Chroma instance
vectorstore = Chroma.from_documents(docs, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Define EmbeddingsFilter with similarity threshold
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.6)

# Wrap base retriever with ContextualCompressionRetriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever
)

compressed_docs = compression_retriever.invoke("How do quantum computers work?")

print(f"Base retriever returned 3 documents, but compressed retriever filtered down to: {len(compressed_docs)} relevant document(s).")
for i, doc in enumerate(compressed_docs, 1):
    print(f"--- Compressed Document {i} ---\n{doc.page_content}\n")


/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base retriever returned 3 documents, but compressed retriever filtered down to: 2 relevant document(s).
--- Compressed Document 1 ---
Quantum computing leverages superposition and entanglement to solve complex optimization problems.

--- Compressed Document 2 ---
Classical computers process binary bits 0 or 1 using silicon transistors.

